# Black-Litterman Portfolio Optimization Engine

**Author:** (your name here) · **Type:** Quantitative Finance / Portfolio Construction
**Stack:** Python, yfinance, NumPy, SciPy, scikit-learn, pandas, matplotlib, seaborn

---

## 1. Project Overview

This project implements the **Black-Litterman (BL) model**, the industry-standard
Bayesian framework for combining the *market's* implied view of expected returns
with an *investor's* subjective views, to produce a more stable, intuitive set of
expected returns than raw historical-mean estimation (which is notoriously noisy
and produces wild, unstable portfolio weights in classic Markowitz mean-variance
optimization).

The pipeline:
1. Pull real price + market-cap data for a basket of stocks via `yfinance`.
2. Reverse-engineer the market's *implied equilibrium returns* from market-cap
   weights and the covariance matrix (this is the BL "prior").
3. Inject investor views (absolute and relative) with confidence levels.
4. Blend prior + views via the BL posterior formula to get new expected returns
   and a posterior covariance matrix.
5. Run mean-variance optimization (max Sharpe) on the BL-blended estimates.
6. Compare against naive historical-mean optimization and market-cap weights.
7. Visualize the efficient frontier, weight allocations, and risk metrics.

## 2. Real-World Finance Use Case

Hedge funds and asset managers (Goldman Sachs literally invented this in 1990)
use Black-Litterman because plugging raw historical average returns into a
Markowitz optimizer produces concentrated, unstable, often nonsensical portfolios
(e.g. 80% in one stock because its trailing average happened to be high). BL fixes
this by anchoring expected returns to the market-implied equilibrium (what
prices already encode) and only nudging them based on views you actually hold
conviction in, scaled by how confident you are. This is exactly how
institutional quant teams turn analyst opinions ("we think NVDA will outperform")
into actual portfolio weights without blowing up the portfolio.

## 3. System Architecture

```
 ┌────────────────┐     ┌──────────────────┐     ┌───────────────────┐
 │  Data Layer     │ --> │  Estimation Layer │ --> │  Optimization Layer│
 │ (yfinance pull, │     │ (cov shrinkage,   │     │ (BL posterior,     │
 │  mkt caps, rf)  │     │  implied returns) │     │  max-Sharpe, EF)   │
 └────────────────┘     └──────────────────┘     └───────────────────┘
                                                          │
                                                          v
                                                ┌───────────────────┐
                                                │  Reporting Layer   │
                                                │ (charts, metrics,  │
                                                │  CSV export)       │
                                                └───────────────────┘
```

## 4. Required APIs and Data Sources
- **Yahoo Finance** via `yfinance` — adjusted close prices, market caps, 13-week T-bill (`^IRX`) as risk-free proxy.
- No paid API key required (this is intentional, so the project runs end-to-end for free in Colab).

## 5. Required Python Libraries
`yfinance`, `numpy`, `pandas`, `scipy`, `scikit-learn`, `matplotlib`, `seaborn`

## 6. Folder/File Structure (conceptual — even though this runs in one Colab notebook)
```
black-litterman-optimizer/
├── README.md
├── black_litterman.ipynb        <- this notebook
├── /data
│    └── prices_cache.csv        <- optional cached pull
├── /outputs
│    ├── bl_weights.csv
│    ├── efficient_frontier.png
│    ├── weights_comparison.png
│    └── correlation_heatmap.png
└── requirements.txt
```
Each section below is laid out as its own Colab cell so you can literally
recreate that folder structure by running cells top to bottom — exported files
land in `/content/outputs/` automatically.


## 7. Step 0 — Install dependencies
Colab pre-installs most of these, but `yfinance` usually needs a fresh install.

In [ ]:
!pip install yfinance --quiet --upgrade
print("Dependencies installed.")

## Step 1 — Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import json
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
from sklearn.covariance import LedoitWolf
from datetime import datetime, timedelta

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

os.makedirs("/content/outputs", exist_ok=True)
print("Imports ready.")

## Step 2 — Configuration

Everything tunable lives in one place. Swap tickers, lookback window, or views
here without touching the engine code below.

In [ ]:
CONFIG = {
    "tickers": ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA",
                "JPM", "JNJ", "XOM", "PG", "V"],
    "market_ticker": "SPY",        # proxy for "the market" used to derive implied returns
    "risk_free_ticker": "^IRX",    # 13-week T-bill yield, annualized %, used as risk-free rate
    "lookback_years": 5,
    "tau": 0.05,                   # BL scalar — uncertainty of the prior (typically 0.01-0.05)
    # Investor views: each is either "absolute" (single asset) or
    # "relative" (asset A outperforms asset B by X% annually).
    # confidence in (0, 1]: 1.0 = certain, lower = weaker conviction.
    "views": [
        {"type": "absolute", "assets": ["NVDA"],          "return": 0.25, "confidence": 0.60},
        {"type": "relative", "assets": ["JPM", "XOM"],     "return": 0.05, "confidence": 0.50},
        {"type": "absolute", "assets": ["AAPL"],           "return": 0.12, "confidence": 0.70},
    ],
}

END_DATE = datetime.today()
START_DATE = END_DATE - timedelta(days=365 * CONFIG["lookback_years"])
print(f"Universe: {CONFIG['tickers']}")
print(f"Window:   {START_DATE.date()} -> {END_DATE.date()}")

## 8. Step 3 — Data Collection Pipeline

Robust download with error handling: retries are not needed for a single batch
call, but we validate the result, warn on missing tickers, and forward-fill
small gaps (e.g. exchange holidays that don't line up across assets).

In [ ]:
def download_price_data(tickers, start, end):
    """Download adjusted close prices for a list of tickers. Raises if the
    download fails outright; warns (but continues) if some tickers come back
    empty, dropping them from the universe."""
    try:
        raw = yf.download(tickers, start=start, end=end,
                           auto_adjust=True, progress=False)
    except Exception as e:
        raise RuntimeError(f"Price download failed: {e}")

    if raw.empty:
        raise RuntimeError("No price data returned — check tickers/dates/network.")

    prices = raw["Close"] if isinstance(raw.columns, pd.MultiIndex) else raw[["Close"]]
    prices = prices.dropna(axis=1, how="all")

    missing = sorted(set(tickers) - set(prices.columns))
    if missing:
        print(f"WARNING: no data for {missing}, dropping from universe.")

    prices = prices.ffill().dropna()
    if prices.shape[0] < 60:
        raise RuntimeError("Fewer than 60 trading days of overlapping data — widen the date range.")
    return prices


def get_market_caps(tickers):
    """Fetch market caps for weighting. Falls back to equal weighting for any
    ticker whose market cap can't be retrieved, rather than crashing."""
    caps = {}
    for t in tickers:
        try:
            info = yf.Ticker(t).get_info()
            cap = info.get("marketCap")
            caps[t] = float(cap) if cap else np.nan
        except Exception:
            caps[t] = np.nan

    s = pd.Series(caps)
    if s.isna().all():
        print("WARNING: no market caps retrieved — falling back to equal weights.")
        s[:] = 1.0
    elif s.isna().any():
        print(f"WARNING: missing market cap for {list(s[s.isna()].index)}, using sector-average fallback.")
        s = s.fillna(s.mean())

    return s / s.sum()


def get_risk_free_rate(ticker="^IRX"):
    """Annualized risk-free rate from 13-week T-bill yield. Falls back to a
    sane default if the data feed hiccups."""
    try:
        data = yf.download(ticker, period="5d", progress=False)["Close"]
        rf = float(data.dropna().iloc[-1]) / 100.0
        if not (0 <= rf <= 0.25):
            raise ValueError("implausible rf value")
        return rf
    except Exception:
        print("WARNING: risk-free rate fetch failed, defaulting to 4%.")
        return 0.04


tickers = CONFIG["tickers"]
prices = download_price_data(tickers + [CONFIG["market_ticker"]], START_DATE, END_DATE)
tickers = [t for t in tickers if t in prices.columns]  # keep only what actually downloaded

asset_prices = prices[tickers]
market_prices = prices[CONFIG["market_ticker"]]

market_caps = get_market_caps(tickers)
rf = get_risk_free_rate(CONFIG["risk_free_ticker"])

print(f"\nFinal universe ({len(tickers)} assets): {tickers}")
print(f"Risk-free rate: {rf:.2%}")
market_caps.sort_values(ascending=False)

## 9. Step 4 — Cleaning & Feature Engineering

Convert prices to daily returns, then build two covariance estimates:
- **Sample covariance** (the textbook estimator)
- **Ledoit-Wolf shrinkage covariance** — pulls the noisy sample covariance toward
  a more stable structured target, which is standard practice because raw
  sample covariance is poorly conditioned with limited history relative to the
  number of assets.

In [ ]:
asset_returns = asset_prices.pct_change().dropna()
market_returns = market_prices.pct_change().dropna()

mu_hist = asset_returns.mean() * 252                 # naive annualized historical mean
cov_sample = asset_returns.cov() * 252

lw = LedoitWolf().fit(asset_returns.values)
cov_shrunk = pd.DataFrame(lw.covariance_ * 252, index=tickers, columns=tickers)

print("Shrinkage intensity (Ledoit-Wolf):", round(lw.shrinkage_, 4))
print("\nAnnualized historical mean returns:")
mu_hist.sort_values(ascending=False).map(lambda x: f"{x:.2%}")

## 10. Step 5 — Market-Implied Equilibrium Returns (the BL "Prior")

This is the heart of Black-Litterman: instead of trusting noisy historical
averages, we **reverse-engineer** what expected returns *must be*, given that
the market already holds these assets in market-cap-weighted proportions and is
risk-averse with risk-aversion coefficient δ.

```
δ = (E[R_market] - r_f) / Var(R_market)
π = δ · Σ · w_market
```

`π` (pi) is the prior expected excess return vector everything else gets
blended with.

In [ ]:
market_mean_ret = market_returns.mean() * 252
market_var = market_returns.var() * 252

delta = (market_mean_ret - rf) / market_var   # market risk-aversion coefficient
w_mkt = market_caps[tickers].values

pi = delta * cov_shrunk.values @ w_mkt        # implied equilibrium EXCESS returns

print(f"Market risk-aversion (delta): {delta:.3f}")
pi_series = pd.Series(pi, index=tickers)
print("\nImplied equilibrium excess returns (the BL prior):")
pi_series.sort_values(ascending=False).map(lambda x: f"{x:.2%}")

## 11. Step 6 — Investor Views → P, Q, Ω

Each view becomes one row of the "pick" matrix `P`, one entry of the target
vector `Q`, and one diagonal entry of the uncertainty matrix `Ω` (Omega).
Lower confidence → larger Ω entry → that view gets blended in more weakly.

In [ ]:
def build_views(views, tickers, tau, cov):
    """Translate human-readable views into the (P, Q, Omega) matrices BL needs."""
    asset_idx = {t: i for i, t in enumerate(tickers)}
    n, k = len(tickers), len(views)
    P = np.zeros((k, n))
    Q = np.zeros(k)
    confidences = np.zeros(k)

    for i, v in enumerate(views):
        missing = [a for a in v["assets"] if a not in asset_idx]
        if missing:
            raise ValueError(f"View references unknown tickers {missing} (dropped from universe?)")

        if v["type"] == "absolute":
            P[i, asset_idx[v["assets"][0]]] = 1.0
            Q[i] = v["return"]
        elif v["type"] == "relative":
            P[i, asset_idx[v["assets"][0]]] = 1.0
            P[i, asset_idx[v["assets"][1]]] = -1.0
            Q[i] = v["return"]
        else:
            raise ValueError(f"Unknown view type: {v['type']}")

        c = np.clip(v["confidence"], 1e-4, 0.9999)
        confidences[i] = c

    tau_cov = tau * cov.values
    omega_diag = np.array([
        (P[i] @ tau_cov @ P[i].T) * (1.0 / confidences[i] - 1.0)
        for i in range(k)
    ])
    Omega = np.diag(omega_diag)
    return P, Q, Omega


P, Q, Omega = build_views(CONFIG["views"], tickers, CONFIG["tau"], cov_shrunk)
print("View matrix P:\n", pd.DataFrame(P, columns=tickers))
print("\nView targets Q:", Q)
print("\nView uncertainty (Omega diagonal):", np.diag(Omega))

## 12a. Step 7 — Black-Litterman Posterior Engine

The closed-form BL posterior:

```
M     = [(τΣ)⁻¹ + PᵗΩ⁻¹P]⁻¹
μ_BL  = M · [(τΣ)⁻¹π + PᵗΩ⁻¹Q]
Σ_BL  = Σ + M
```

`μ_BL` is the blended expected-return vector; `Σ_BL` is the posterior
covariance, which is slightly larger than the sample covariance because it
also reflects estimation uncertainty in the mean.

In [ ]:
def black_litterman_posterior(pi, cov, tau, P, Q, Omega):
    """Compute BL posterior expected returns and covariance."""
    cov_arr = cov.values if isinstance(cov, pd.DataFrame) else cov
    tau_cov = tau * cov_arr

    try:
        inv_tau_cov = np.linalg.inv(tau_cov)
        inv_omega = np.linalg.inv(Omega)
    except np.linalg.LinAlgError as e:
        raise RuntimeError(f"Singular matrix in BL computation: {e}")

    M_inv = inv_tau_cov + P.T @ inv_omega @ P
    M = np.linalg.inv(M_inv)
    mu_bl = M @ (inv_tau_cov @ pi + P.T @ inv_omega @ Q)
    cov_bl = cov_arr + M
    return mu_bl, cov_bl


mu_bl, cov_bl = black_litterman_posterior(pi, cov_shrunk, CONFIG["tau"], P, Q, Omega)
mu_bl_series = pd.Series(mu_bl, index=tickers)

comparison = pd.DataFrame({
    "Historical Mean": mu_hist,
    "Market-Implied (Prior)": pi_series,
    "Black-Litterman (Posterior)": mu_bl_series,
}).map(lambda x: f"{x:.2%}")
comparison

## 12b. Step 8 — Portfolio Optimization (Max Sharpe)

Standard mean-variance optimization, but run three times on three different
expected-return estimates so we can see exactly what the BL blend changes
relative to naive historical means and the raw market-cap weights.

In [ ]:
def optimize_max_sharpe(mu, cov, bounds=None):
    """Long-only max-Sharpe portfolio via SLSQP."""
    n = len(mu)
    bounds = bounds or [(0.0, 1.0)] * n
    constraints = ({"type": "eq", "fun": lambda w: np.sum(w) - 1.0},)

    def neg_sharpe(w):
        ret = w @ mu
        vol = np.sqrt(max(w @ cov @ w, 1e-12))
        return -ret / vol

    w0 = np.repeat(1.0 / n, n)
    result = minimize(neg_sharpe, w0, method="SLSQP",
                       bounds=bounds, constraints=constraints,
                       options={"maxiter": 1000, "ftol": 1e-10})
    if not result.success:
        raise RuntimeError(f"Optimization failed: {result.message}")
    w = np.clip(result.x, 0, None)
    return w / w.sum()


w_bl = optimize_max_sharpe(mu_bl, cov_bl.values if isinstance(cov_bl, pd.DataFrame) else cov_bl)
w_hist = optimize_max_sharpe(mu_hist.values, cov_sample.values)
w_market = w_mkt / w_mkt.sum()

weights_df = pd.DataFrame({
    "Market-Cap": w_market,
    "Naive Historical": w_hist,
    "Black-Litterman": w_bl,
}, index=tickers)

weights_df.map(lambda x: f"{x:.1%}")

## Step 9 — Performance Metrics

In [ ]:
def portfolio_performance(weights, mu, cov, label=""):
    ret = weights @ mu
    vol = np.sqrt(weights @ cov @ weights)
    sharpe = ret / vol if vol > 0 else np.nan
    print(f"{label:<22} | Return: {ret:7.2%} | Vol: {vol:7.2%} | Sharpe: {sharpe:5.2f}")
    return {"return": ret, "vol": vol, "sharpe": sharpe}


cov_bl_arr = cov_bl.values if isinstance(cov_bl, pd.DataFrame) else cov_bl

print("Performance evaluated on the SAME (Black-Litterman) risk/return model")
print("so the only thing that varies is the weight allocation strategy:\n")
perf_market = portfolio_performance(w_market, mu_bl, cov_bl_arr, "Market-Cap weights")
perf_hist   = portfolio_performance(w_hist,   mu_bl, cov_bl_arr, "Naive Historical weights")
perf_bl     = portfolio_performance(w_bl,     mu_bl, cov_bl_arr, "Black-Litterman weights")

metrics_df = pd.DataFrame([perf_market, perf_hist, perf_bl],
                          index=["Market-Cap", "Naive Historical", "Black-Litterman"])
metrics_df

## 11. Visualizations & Dashboard Components

Four charts: weight allocation comparison, the efficient frontier with all
three portfolios marked, the asset correlation structure, and a clean pie
chart of the final recommended (BL) allocation.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
weights_df.plot(kind="bar", ax=ax, width=0.75)
ax.set_title("Portfolio Weight Allocation by Method", fontsize=13, fontweight="bold")
ax.set_ylabel("Weight")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.legend(title="Method")
plt.tight_layout()
plt.savefig("/content/outputs/weights_comparison.png", dpi=150)
plt.show()

In [ ]:
def efficient_frontier(mu, cov, n_points=40, bounds=None):
    n = len(mu)
    bounds = bounds or [(0.0, 1.0)] * n
    targets = np.linspace(mu.min(), mu.max(), n_points)
    vols, rets = [], []

    for target in targets:
        constraints = (
            {"type": "eq", "fun": lambda w: np.sum(w) - 1.0},
            {"type": "eq", "fun": lambda w, t=target: w @ mu - t},
        )
        w0 = np.repeat(1.0 / n, n)
        res = minimize(lambda w: np.sqrt(max(w @ cov @ w, 1e-12)), w0,
                        method="SLSQP", bounds=bounds, constraints=constraints,
                        options={"maxiter": 1000})
        if res.success:
            vols.append(res.fun)
            rets.append(target)
    return np.array(vols), np.array(rets)


ef_vols, ef_rets = efficient_frontier(mu_bl, cov_bl_arr)

fig, ax = plt.subplots(figsize=(9, 6.5))
ax.plot(ef_vols, ef_rets, lw=2, color="#2c3e50", label="Efficient Frontier (BL estimates)")

for label, w, color, marker in [
    ("Market-Cap", w_market, "#3498db", "o"),
    ("Naive Historical", w_hist, "#e67e22", "s"),
    ("Black-Litterman", w_bl, "#27ae60", "*"),
]:
    r = w @ mu_bl
    v = np.sqrt(w @ cov_bl_arr @ w)
    ax.scatter(v, r, s=220 if marker == "*" else 140, color=color,
               marker=marker, edgecolor="black", zorder=5, label=label)

ax.set_xlabel("Annualized Volatility")
ax.set_ylabel("Annualized Excess Return")
ax.set_title("Efficient Frontier — Portfolio Comparison", fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.legend()
plt.tight_layout()
plt.savefig("/content/outputs/efficient_frontier.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 7))
corr = asset_returns.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5, ax=ax, cbar_kws={"shrink": 0.8})
ax.set_title("Asset Return Correlation Matrix", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("/content/outputs/correlation_heatmap.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 7.5))
final_weights = weights_df["Black-Litterman"]
shown = final_weights[final_weights > 0.01]   # hide near-zero slivers in the chart
colors = sns.color_palette("Set2", len(shown))
ax.pie(shown.values, labels=shown.index, autopct="%1.1f%%",
       startangle=90, colors=colors, wedgeprops={"edgecolor": "white", "linewidth": 1.5})
ax.set_title("Final Recommended Allocation (Black-Litterman)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("/content/outputs/final_allocation_pie.png", dpi=150)
plt.show()

## 13. Final Deliverables — Export Results

In [ ]:
weights_df.to_csv("/content/outputs/bl_weights.csv")
metrics_df.to_csv("/content/outputs/performance_metrics.csv")
comparison.to_csv("/content/outputs/expected_returns_comparison.csv")

summary = {
    "config": CONFIG,
    "risk_free_rate": rf,
    "market_risk_aversion_delta": float(delta),
    "final_weights": final_weights.round(4).to_dict(),
    "performance": {k: {m: float(v) for m, v in row.items()} for k, row in metrics_df.to_dict("index").items()},
}
with open("/content/outputs/run_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

print("Saved to /content/outputs/:")
for f in sorted(os.listdir("/content/outputs")):
    print(" -", f)

print("\nFinal Black-Litterman recommended portfolio:")
final_weights[final_weights > 0.001].sort_values(ascending=False).map(lambda x: f"{x:.1%}")

## 14. Resume Description

> **Black-Litterman Portfolio Optimization Engine** — Built a quantitative
> portfolio construction tool in Python implementing the Black-Litterman
> Bayesian asset allocation model from first principles (reverse-optimized
> market-implied equilibrium returns, custom view-blending engine with
> confidence-weighted uncertainty matrices, Ledoit-Wolf covariance shrinkage).
> Combined market-cap-weighted equilibrium returns with investor views to
> generate posterior return/risk estimates, then solved constrained
> mean-variance optimization (SciPy/SLSQP) to produce max-Sharpe portfolios.
> Benchmarked against naive historical-mean optimization and market-cap
> weighting across a 10-asset equity universe using live Yahoo Finance data,
> visualizing results via efficient frontier plots, correlation heatmaps, and
> allocation comparisons.

## 15. Potential Upgrades
- Swap the simple Ω construction for **Idzorek's method**, which lets you
  specify confidence as "% tilt toward the view" instead of a raw scalar —
  more intuitive for non-quants.
- Add a **rolling backtest**: re-run BL monthly/quarterly over history and
  track realized portfolio performance vs. a buy-and-hold benchmark.
- Pull views automatically from analyst price targets (e.g. scraped consensus
  targets) instead of hardcoding them, to make the views layer dynamic.
- Add **transaction cost** and **turnover constraints** to the optimizer so
  rebalancing isn't unrealistically frictionless.
- Extend covariance estimation with a **factor model** (Fama-French) instead
  of (or blended with) Ledoit-Wolf shrinkage.
- Wrap the whole thing in a **Streamlit dashboard** so view confidence levels
  can be adjusted with sliders and the frontier updates live.
- Add **bootstrapped confidence intervals** around the BL weights to show how
  sensitive the allocation is to estimation error.
